# Scenár uzávierky — výber ulice, výsledky, mapa

Notebook **nič v modeli nemení**. Len sa pýta bežiaceho API na to, čo už pipeline vypočítala.

## Čo notebook robí

1. Nájde `link_id` podľa názvu ulice
2. Zobrazí baseline mapu (objemy pred uzávierkou)
3. Spustí scenár uzávierky cez API
4. Vypíše tabuľku najviac ovplyvnených úsekov
5. Nakreslí rozdielovú mapu (kam sa doprava presunula)

## Predpoklady

**Notebook nepotrebuje `aequilibrae` ani `geopandas`.** Ťažkú prácu robí server.

```bash
pip install requests pandas folium jupyter
```

Server musí bežať v prostredí, kde je `aequilibrae` (venv s Pythonom 3.12 alebo Docker):

```bash
python run.py --config config/brno/sim.yaml serve
```

> ### ⚠️ Aktuálny stav repozitára
>
> Adresár `outputs/` je **prázdny** — výsledky predchádzajúceho behu tam nie sú.
> Kým sa pipeline nespustí (aspoň po krok `assign`), API vráti `404` a kroky 2–5
> nebudú fungovať.
>
> **Krok 1 (hľadanie ulice) funguje aj teraz** — číta priamo zo SQLite projektu,
> ktorý existuje (`project/brno_aeq/project_database.sqlite`, 36 741 linkov).

---
## Nastavenie

In [ ]:
# --- Nastavenie ---------------------------------------------------------
API_URL    = "http://localhost:8000"
PROJECT_DB = "../project/brno_aeq/project_database.sqlite"   # pre offline hľadanie ulíc
ULICA      = "Jihlavská"                                     # čo chceme zavrieť

import json, sqlite3, time
from pathlib import Path

import pandas as pd

try:
    import requests
except ImportError:
    raise SystemExit("Chýba requests:  pip install requests")

try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    print("folium nie je nainštalovaný — mapy sa preskočia (pip install folium)")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
print("OK")

In [ ]:
# --- Kontrola, či API beží a či má dáta ---------------------------------
def api_health():
    try:
        r = requests.get(f"{API_URL}/api/meta", timeout=5)
    except requests.exceptions.ConnectionError:
        print("❌ API nebeží.")
        print("   Spustite v inom termináli (v prostredí s aequilibrae):")
        print("   python run.py --config config/brno/sim.yaml serve")
        return False
    if r.status_code != 200:
        print(f"⚠️  /api/meta vrátilo {r.status_code}: {r.text[:300]}")
        return False
    print("✅ API beží:", json.dumps(r.json(), ensure_ascii=False)[:300])

    r2 = requests.get(f"{API_URL}/api/links", params={"min_volume": 999999}, timeout=30)
    if r2.status_code == 404:
        print("\n⚠️  Sieť nie je vyexportovaná — chýba network_links.parquet.")
        print("   Detail:", r2.json().get("detail", "")[:300])
        print("   Spustite aspoň:  normalize-network  →  ...  →  assign")
        return False
    print("✅ Sieť je dostupná.")
    return True

API_READY = api_health()

---
## Krok 1 — nájsť úseky ulice

Jedna ulica = veľa `link_id` (úsek medzi každými dvoma križovatkami).
Jihlavská má napríklad **112 linkov**.

Táto bunka funguje **aj bez bežiaceho API** — číta priamo SQLite projektu.

In [ ]:
def najdi_ulicu_sqlite(nazov, db_path=PROJECT_DB, link_types=None):
    """Nájde linky podľa názvu v projektovej SQLite. Nepotrebuje bežiace API."""
    p = Path(db_path)
    if not p.exists():
        raise FileNotFoundError(f"Projektová DB nenájdená: {p.resolve()}")
    cn = sqlite3.connect(str(p))
    try:
        q = ("SELECT link_id, name, osm_ref, link_type, direction, "
             "lanes_ab, lanes_ba, ROUND(distance) AS dlzka_m "
             "FROM links WHERE name LIKE ? ORDER BY link_id")
        df = pd.read_sql_query(q, cn, params=(f"%{nazov}%",))
    finally:
        cn.close()
    if link_types:
        df = df[df["link_type"].isin(link_types)]
    return df.reset_index(drop=True)


useky = najdi_ulicu_sqlite(ULICA)
print(f"Nájdených linkov pre '{ULICA}': {len(useky)}")
print("\nRozdelenie podľa typu cesty:")
print(useky["link_type"].value_counts().to_string())
print(f"\nCelková dĺžka: {useky['dlzka_m'].sum():,.0f} m")
useky.head(20)

In [ ]:
# --- Výber, ktoré linky reálne zavrieme ---------------------------------
# Možnosti:
#   a) celá ulica              -> vyber = useky
#   b) len hlavný ťah          -> filter na link_type
#   c) konkrétny úsek          -> zoznam link_id

vyber = useky[useky["link_type"] == "secondary"].copy()       # (b)
# vyber = useky.copy()                                        # (a)
# vyber = useky[useky["link_id"].isin([35, 36, 37, 38])]      # (c)

ZAVRIET = vyber["link_id"].astype(int).tolist()
print(f"Zavrieme {len(ZAVRIET)} linkov, spolu {vyber['dlzka_m'].sum():,.0f} m")
print(ZAVRIET)

---
## Krok 2 — baseline: čo je na sieti teraz

`GET /api/links` vráti GeoJSON so **skalibrovanými** objemami z posledného behu
(`wd_daily_tot`), plus `VOC_max` a `LOS_max`.

`min_volume` filtruje drobné ulice, aby bola mapa čitateľná a odpoveď menšia.

In [ ]:
def nacitaj_linky(min_volume=1000):
    r = requests.get(f"{API_URL}/api/links", params={"min_volume": min_volume}, timeout=120)
    r.raise_for_status()
    gj = r.json()
    rows = []
    for f in gj["features"]:
        p = dict(f["properties"])
        g = f["geometry"]
        if g and g["type"] == "LineString":
            p["_coords"] = g["coordinates"]           # [[lon, lat], ...]
        elif g and g["type"] == "MultiLineString":
            p["_coords"] = [c for part in g["coordinates"] for c in part]
        else:
            p["_coords"] = None
        rows.append(p)
    return pd.DataFrame(rows)


if API_READY:
    base = nacitaj_linky(min_volume=1000)
    print(f"Načítaných linkov: {len(base):,}")
    cols = [c for c in ["link_id", "name", "link_type", "wd_daily_tot", "VOC_max", "LOS_max"]
            if c in base.columns]
    display(base.nlargest(10, "wd_daily_tot")[cols])
else:
    base = None
    print("Preskočené — API nie je pripravené.")

In [ ]:
# --- Obmedzenie uzávierky na katastrálne územie Brna ---------------------
# Ulica s rovnakým názvom pokračuje aj za hranicou mesta: pôvodný scenár
# zavrel Jihlavskú v rozsahu lon 16.497–16.602, teda vrátane úsekov
# v Troubsku/Popůvkách. Tu ich odfiltrujeme.
#
# Test "je link v Brne" robíme voči interným zónam modelu (mestské časti);
# externé bránové zóny (is_external == 1) sa vynechávajú.
# Bez geopandas/shapely — ray casting na GeoJSON z API.

LEN_BRNO = True          # False = zavrieť celú ulicu vrátane úsekov mimo mesta


def _v_prstenci(x, y, ring):
    inside, n = False, len(ring)
    j = n - 1
    for i in range(n):
        xi, yi = ring[i][0], ring[i][1]
        xj, yj = ring[j][0], ring[j][1]
        if (yi > y) != (yj > y):
            if x < (xj - xi) * (y - yi) / ((yj - yi) or 1e-12) + xi:
                inside = not inside
        j = i
    return inside


def _v_polygone(x, y, poly):
    """poly = zoznam prstencov, [0] je vonkajší, ďalšie sú diery."""
    if not _v_prstenci(x, y, poly[0]):
        return False
    return not any(_v_prstenci(x, y, r) for r in poly[1:])


def nacitaj_polygon_mesta():
    """Interné zóny modelu ako zoznam polygónov (+ ich bbox pre rýchly filter)."""
    r = requests.get(f"{API_URL}/api/zones", timeout=60)
    r.raise_for_status()
    polys = []
    for f in r.json()["features"]:
        if int(f["properties"].get("is_external") or 0) == 1:
            continue
        g = f["geometry"]
        if g["type"] == "Polygon":
            polys.append(g["coordinates"])
        elif g["type"] == "MultiPolygon":
            polys.extend(g["coordinates"])
    boxy = []
    for p in polys:
        xs = [c[0] for c in p[0]]
        ys = [c[1] for c in p[0]]
        boxy.append((min(xs), min(ys), max(xs), max(ys)))
    return polys, boxy


def je_v_meste(coords, polys, boxy):
    """Testuje stredový vrchol linky (robustnejšie než ťažisko pri zakrivenej ulici)."""
    if not coords:
        return False
    x, y = coords[len(coords) // 2]
    for p, (x0, y0, x1, y1) in zip(polys, boxy):
        if x0 <= x <= x1 and y0 <= y <= y1 and _v_polygone(x, y, p):
            return True
    return False


if API_READY and LEN_BRNO:
    _polys, _boxy = nacitaj_polygon_mesta()
    print(f"Interných zón (mestské časti): {len(_polys)}")

    _geom = nacitaj_linky(min_volume=0).set_index("link_id")["_coords"].to_dict()

    v_meste, mimo = [], []
    for lid in ZAVRIET:
        (v_meste if je_v_meste(_geom.get(lid), _polys, _boxy) else mimo).append(lid)

    print(f"\n'{ULICA}' — linkov na zavretie: {len(ZAVRIET)}")
    print(f"  v Brne-mesto: {len(v_meste)}")
    print(f"  mimo mesta:   {len(mimo)}  -> vynechané: {mimo}")

    vyber = vyber[vyber["link_id"].astype(int).isin(v_meste)].copy()
    ZAVRIET = v_meste
    print(f"\nZavrieme {len(ZAVRIET)} linkov, spolu {vyber['dlzka_m'].sum():,.0f} m")
else:
    print("Preskočené — beží sa na celej ulici (LEN_BRNO=False alebo API nie je pripravené).")

In [ ]:
def sirka(hodnota, maximum, min_w=1.0, max_w=9.0):
    """Hrúbka čiary úmerná hodnote."""
    if not maximum or pd.isna(hodnota):
        return min_w
    return min_w + (max_w - min_w) * min(abs(hodnota) / maximum, 1.0)


def farba_los(los):
    return {"A": "#2c7fb8", "B": "#41b6c4", "C": "#a1dab4",
            "D": "#fecc5c", "E": "#fd8d3c", "F": "#e31a1c"}.get(str(los), "#999999")


def stred_mapy(df):
    lats, lons = [], []
    for c in df["_coords"].dropna().head(400):
        for lon, lat in c[::10] or c:
            lats.append(lat); lons.append(lon)
    if not lats:
        return (49.195, 16.606)          # Brno
    return (sum(lats) / len(lats), sum(lons) / len(lons))


def mapa_baseline(df, zvyraznit=None, zoom=12):
    m = folium.Map(location=stred_mapy(df), zoom_start=zoom, tiles="CartoDB positron")
    vmax = float(df["wd_daily_tot"].quantile(0.98))
    zvyraznit = set(zvyraznit or [])

    vrstva = folium.FeatureGroup(name="Baseline — denné objemy")
    for _, r in df.iterrows():
        if not r["_coords"]:
            continue
        folium.PolyLine(
            [(lat, lon) for lon, lat in r["_coords"]],
            color=farba_los(r.get("LOS_max")),
            weight=sirka(r.get("wd_daily_tot", 0), vmax),
            opacity=0.75,
            tooltip=(f"<b>{r.get('name') or '(bez názvu)'}</b><br>"
                     f"link_id: {r['link_id']}<br>"
                     f"objem: {r.get('wd_daily_tot', 0):,.0f} voz/deň<br>"
                     f"V/C: {r.get('VOC_max', 0):.2f} &nbsp; LOS: {r.get('LOS_max')}"),
        ).add_to(vrstva)
    vrstva.add_to(m)

    if zvyraznit:
        vz = folium.FeatureGroup(name="Plánovaná uzávierka")
        for _, r in df[df["link_id"].isin(zvyraznit)].iterrows():
            if not r["_coords"]:
                continue
            folium.PolyLine(
                [(lat, lon) for lon, lat in r["_coords"]],
                color="#000000", weight=11, opacity=0.9,
                tooltip=f"ZAVRETÉ — link_id {r['link_id']}",
            ).add_to(vz)
        vz.add_to(m)

    folium.LayerControl().add_to(m)
    return m


if API_READY and HAS_FOLIUM:
    display(mapa_baseline(base, zvyraznit=ZAVRIET))
else:
    print("Mapa preskočená.")

**Farba = LOS** (A modrá voľno … F červená kolóna), **hrúbka = denný objem**.
Čierne čiary sú linky, ktoré ideme zavrieť.

---
## Krok 3 — spustiť scenár

`POST /api/scenarios/run` spustí **celé nové priradenie** (užívateľské ekvilibrium)
so zavretými linkami. Beží na pozadí, rádovo minúty.

Zavretému linku sa v pamäti nastaví kapacita `0.001` a čas `99 999 s`, takže ho
BPR urobí prohibitívne drahým. **Projektová databáza sa nemení.**

In [ ]:
def spusti_scenar(link_ids, lanes_map=None, typ="full", zostane_pruhov=0, smer="both"):
    """typ='full' = úplná uzávierka, typ='lanes' = redukcia pruhov."""
    lanes_map = lanes_map or {}
    links = [{
        "link_id": int(lid),
        "direction": smer,
        "closure_type": typ,
        "lanes": max(int(lanes_map.get(int(lid), 2)), 1),
        "lanes_remaining": int(zostane_pruhov),
    } for lid in link_ids]

    r = requests.post(f"{API_URL}/api/scenarios/run",
                      json={"links": links}, timeout=60)
    if r.status_code not in (200, 202):
        raise RuntimeError(f"{r.status_code}: {r.text[:500]}")
    return r.json()["id"]


def cakaj(job_id, timeout_s=1800, interval_s=5):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        st = requests.get(f"{API_URL}/api/scenarios/{job_id}/status", timeout=15).json()
        print(f"\r  {st['status']}  ({st['elapsed_seconds']:.0f} s)   ", end="")
        if st["status"] == "done":
            print("\n✅ hotovo")
            return st
        if st["status"] == "error":
            raise RuntimeError(f"Scenár zlyhal: {st.get('error')}")
        time.sleep(interval_s)
    raise TimeoutError("Scenár neskončil v limite.")


if API_READY:
    lanes_map = dict(zip(vyber["link_id"].astype(int), vyber["lanes_ab"].fillna(2).astype(int)))
    JOB_ID = spusti_scenar(ZAVRIET, lanes_map=lanes_map, typ="full")
    print("job:", JOB_ID)
    cakaj(JOB_ID)
else:
    JOB_ID = None
    print("Preskočené.")

---
## Krok 4 — tabuľka výsledkov

`GET /api/scenarios/{id}/delta-summary` vráti agregát: koľko linkov sa zmenilo,
priemerná zmena, a top 20 najviac ovplyvnených úsekov.

In [ ]:
if JOB_ID:
    s = requests.get(f"{API_URL}/api/scenarios/{JOB_ID}/delta-summary", timeout=60).json()

    print(f"Ovplyvnených linkov:        {s['total_links_affected']:,}")
    print(f"Priemerná |zmena| objemu:   {s['mean_abs_delta_vol']:,.0f} voz/deň")
    print(f"Priemerná |zmena| v %:      {s['mean_abs_delta_pct']:.1f} %")
    if s.get("max_increase"):
        mi = s["max_increase"]
        print(f"\nNajväčší nárast:  {mi['name'] or '(bez názvu)'} "
              f"{mi['baseline_vol']:,.0f} → {mi['scenario_vol']:,.0f} "
              f"({mi['delta_vol']:+,.0f})")
    if s.get("max_decrease"):
        md = s["max_decrease"]
        print(f"Najväčší pokles:  {md['name'] or '(bez názvu)'} "
              f"{md['baseline_vol']:,.0f} → {md['scenario_vol']:,.0f} "
              f"({md['delta_vol']:+,.0f})")

    top = pd.DataFrame(s["top_affected"])
    if not top.empty:
        top = top.rename(columns={
            "name": "ulica", "link_type": "typ", "baseline_vol": "pred",
            "scenario_vol": "po", "delta_vol": "zmena", "delta_pct": "zmena_%",
            "delta_voc": "zmena_V/C",
        })
    display(top)

In [ ]:
# --- Kontrola: naozaj sa na zavretých linkoch objem vynuloval? -----------
if JOB_ID:
    gj = requests.get(f"{API_URL}/api/scenarios/{JOB_ID}/results", timeout=300).json()
    rows = []
    for f in gj["features"]:
        p = dict(f["properties"])
        g = f["geometry"]
        if g and g["type"] == "LineString":
            p["_coords"] = g["coordinates"]
        elif g and g["type"] == "MultiLineString":
            p["_coords"] = [c for part in g["coordinates"] for c in part]
        else:
            p["_coords"] = None
        rows.append(p)
    res = pd.DataFrame(rows)
    print(f"Linkov vo výsledku: {len(res):,}")

    zavrete = res[res["link_id"].isin(ZAVRIET)]
    print("\nSanity check — zavreté linky:")
    print(f"  objem pred:  {zavrete['baseline_vol'].sum():,.0f} voz/deň")
    print(f"  objem po:    {zavrete['wd_daily_tot'].sum():,.0f} voz/deň")
    print("  (druhé číslo má byť ~0; ak nie je, časť linkov sa nezavrela)")

    print(f"\nSystémová bilancia:")
    print(f"  celkový objem pred: {res['baseline_vol'].sum():,.0f}")
    print(f"  celkový objem po:   {res['wd_daily_tot'].sum():,.0f}")
    print("  (rozdiel = zmena celkovej prejazdenej vzdialenosti, doprava nezmizne)")

---
## Krok 5 — rozdielová mapa

**Červená = pribudlo, modrá = ubudlo**, hrúbka = veľkosť zmeny.
Toto je odpoveď na otázku *„kam sa doprava presunie"*.

In [ ]:
def mapa_rozdielu(res, zavrete_ids, min_delta=100, zoom=12):
    d = res[res["delta_vol"].abs() >= min_delta].copy()
    if "low_volume_flag" in d.columns:
        d = d[d["low_volume_flag"] != 1]        # odfiltruje šum na tichých uliciach
    print(f"Zobrazených linkov so zmenou ≥ {min_delta} voz/deň: {len(d):,}")

    m = folium.Map(location=stred_mapy(d if len(d) else res), zoom_start=zoom,
                   tiles="CartoDB positron")
    vmax = float(d["delta_vol"].abs().quantile(0.95)) if len(d) else 1.0

    up   = folium.FeatureGroup(name="Nárast (červená)")
    down = folium.FeatureGroup(name="Pokles (modrá)")
    for _, r in d.iterrows():
        if not r["_coords"]:
            continue
        dv = float(r["delta_vol"])
        folium.PolyLine(
            [(lat, lon) for lon, lat in r["_coords"]],
            color="#d7191c" if dv > 0 else "#2c7bb6",
            weight=sirka(dv, vmax, 1.5, 10.0),
            opacity=0.8,
            tooltip=(f"<b>{r.get('name') or '(bez názvu)'}</b><br>"
                     f"link_id: {r['link_id']}<br>"
                     f"pred: {r.get('baseline_vol', 0):,.0f} → po: {r.get('wd_daily_tot', 0):,.0f}<br>"
                     f"<b>zmena: {dv:+,.0f} voz/deň ({r.get('delta_pct', 0):+.1f} %)</b><br>"
                     f"V/C po: {r.get('VOC_max', 0):.2f}"),
        ).add_to(up if dv > 0 else down)
    down.add_to(m); up.add_to(m)

    vz = folium.FeatureGroup(name="Uzávierka")
    for _, r in res[res["link_id"].isin(zavrete_ids)].iterrows():
        if not r["_coords"]:
            continue
        folium.PolyLine([(lat, lon) for lon, lat in r["_coords"]],
                        color="#000000", weight=11, opacity=0.95,
                        tooltip=f"ZAVRETÉ — link_id {r['link_id']}").add_to(vz)
    vz.add_to(m)

    folium.LayerControl().add_to(m)
    return m


if JOB_ID and HAS_FOLIUM:
    mapa = mapa_rozdielu(res, ZAVRIET, min_delta=100)
    mapa.save("scenar_rozdiel.html")
    print("Uložené aj ako scenar_rozdiel.html")
    display(mapa)

---
## Voliteľné — porovnanie viacerých variantov

Napr. úplná uzávierka vs. redukcia na 1 pruh.

---
## Poznámky a limity

**Čo notebook robí a nerobí**

- Nemení projektovú databázu ani žiadny súbor pipeline. Scenár beží v pamäti servera.
- Výsledky sa ukladajú do `outputs/scenarios/<job_id>.geojson` a po hodine sa mažú
  (`JOB_TTL_SECONDS` v `src/sim/scenarios/engine.py`). Ak si chcete výsledok
  nechať, uložte si `res` do parquet.
- Scenáre bežia **po jednom** (`ThreadPoolExecutor(max_workers=1)`), takže viac
  variantov ide sériovo.

**Ako čítať čísla**

- Objemy sú **denné** (`wd_daily_tot`), nie hodinové. Na špičkovú hodinu slúži
  `peak_hour_vol_AB/BA`.
- `low_volume_flag = 1` označuje linky, kde je pred aj po menej než 200 voz/deň —
  tam sú percentuálne zmeny šum.
- Celkový objem po uzávierke býva **vyšší** než pred ňou. Nie je to chyba —
  doprava jazdí obchádzkami, takže prejde viac úsekov.

**Rozsah uzávierky**

Ulica s rovnakým názvom často pokračuje aj za hranicou mesta. Bunka „Obmedzenie
uzávierky na katastrálne územie Brna" (`LEN_BRNO = True`) necháva len úseky
vnútri interných zón modelu. Pri Jihlavskej to je podstatné — pôvodný scenár
zavieral aj úseky v Troubsku a Popůvkách, teda ~3 km cesty II/602 mimo Brna.

**Pozor pri Jihlavskej konkrétne**

Jihlavská má `osm_ref = 602`. Cesta 602 **do augusta 2026 nemala bránu** (externá
doprava nemala kadiaľ na ňu vojsť) a zároveň bola vylúčená z kalibrácie aj
validácie. Od zmeny P0-1/P0-2 v `config/brno/sim.yaml` má II/602 vlastnú bránu
a vylúčenia sú zrušené — modelové objemy na nej sú teda po novom porovnávané
s CSD. Ak porovnávate scenár so starším behom, nie sú to porovnateľné baseliny;
pozri `PLAN_ZLEPSENIA.md`.

**Ak chcete porovnať dve verzie modelu**

Spustite ten istý scenár pred aj po zmene dát, uložte oba `res` a porovnajte
`delta_vol` na tých istých `link_id`. Podrobnejšie v `KAMDOJIZDIME_PLAN.md`,
sekcia „Ako porovnávať — tri úrovne".

---
## Poznámky a limity

**Čo notebook robí a nerobí**

- Nemení projektovú databázu ani žiadny súbor pipeline. Scenár beží v pamäti servera.
- Výsledky sa ukladajú do `outputs/scenarios/<job_id>.geojson` a po hodine sa mažú
  (`JOB_TTL_SECONDS` v `src/sim/scenarios/engine.py`). Ak si chcete výsledok
  nechať, uložte si `res` do parquet.
- Scenáre bežia **po jednom** (`ThreadPoolExecutor(max_workers=1)`), takže viac
  variantov ide sériovo.

**Ako čítať čísla**

- Objemy sú **denné** (`wd_daily_tot`), nie hodinové. Na špičkovú hodinu slúži
  `peak_hour_vol_AB/BA`.
- `low_volume_flag = 1` označuje linky, kde je pred aj po menej než 200 voz/deň —
  tam sú percentuálne zmeny šum.
- Celkový objem po uzávierke býva **vyšší** než pred ňou. Nie je to chyba —
  doprava jazdí obchádzkami, takže prejde viac úsekov.

**Pozor pri Jihlavskej konkrétne**

Jihlavská má `osm_ref = 602` a cesta 602 je v Brne vylúčená z kalibrácie
(`exclude_csd_roads` a `csd_validation_exclude_sil` v `config/brno/sim.yaml`).
Modelové objemy na nej teda **nikdy neboli ladené voči CSD**. Relatívne zmeny
(kam sa doprava presunie) sú použiteľné, absolútne čísla berte opatrne.

**Ak chcete porovnať dve verzie modelu**

Spustite ten istý scenár pred aj po zmene dát, uložte oba `res` a porovnajte
`delta_vol` na tých istých `link_id`. Podrobnejšie v `KAMDOJIZDIME_PLAN.md`,
sekcia „Ako porovnávať — tri úrovne".